# CT Zoning Restrictiveness — NZA-Aligned Scorecard + Secondary Composite Score

This notebook was restructured after cross-checking against the National Zoning Atlas (NZA). Two things came out of that cross-check:

1. NZA's own team has publicly argued against single composite zoning-restrictiveness scores (see their "It's Time to Retire the Wharton Index" critique) — they deliberately publish **disaggregated land-area shares by housing type**, not one blended number, because a composite score abstracts away the specific thing a policymaker actually needs to act on.
2. Our original model was structurally the kind of thing that critique targets: a single weighted composite built from somewhat subjective variable-to-restrictiveness mappings.

**So this version produces two outputs, in this priority order:**
- **Primary: an NZA-style scorecard** — for each housing type (1-Family, 2-Family, Multifamily, ADU), what share of *districts* classify as Allowed (as-of-right) / May Be Allowed (discretionary review) / Prohibited, plus NZA's own minimum-lot-size brackets and a binary parking-mandate flag. This is the defensible, NZA-comparable output.
- **Secondary: our original 5-index composite Restrictiveness Score (0-100)**, kept because it's genuinely useful for *ranking and prioritizing* districts, but now explicitly labeled as our own synthesis on top of the scorecard, not an NZA methodology or an NZA-equivalent score.

**Honest limitation carried over from the NZA cross-check:** NZA weights every statistic by *land area in acres*. Our source dataset has no per-district acreage column — only dimensional *requirements* (min lot size, setbacks, etc.), which is a different thing entirely from "how many acres of the town this district covers." Everything below is therefore weighted **equally per district**, not by land area, and that's flagged everywhere it matters rather than quietly presented as land-area-weighted. A true land-area-weighted version would need GIS parcel data this project doesn't have yet.

**Also carried over:** the row-misalignment fix for Bridgeport and the two inference fallbacks (approval pathway, design hearing) from the prior revision — those were genuine data-quality fixes, not methodology changes, so they stay.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../data/collected_data/CT_Zoning_Combined.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} district rows across {df['City'].nunique()} towns.")
df.head(3)

## 0. Data quality fix: row-misalignment bug found in Bridgeport (carried over)

23 of Bridgeport's 25 district rows have a 20-column misalignment — one blank cell is missing somewhere among the Street-Side/Accessory-yard columns, which shifts every value from `Accessory_Max_Size_SqFt` through `P1_Approval_Pathway_Multifamily` one column early. Confirmed by the shifted values landing in semantically wrong columns (a square-footage field holding `"Yes (upper)"`) and by the corrected values matching this session's independent research for Bridgeport DX1 exactly. Only the 23 affected Bridgeport rows are touched.

In [ ]:
_cols = list(df.columns)
_shift_start = _cols.index("Accessory_Max_Size_SqFt")
_shift_end = _cols.index("P1_Approval_Pathway_Multifamily")
_shift_window = _cols[_shift_start:_shift_end + 1]  # 20 columns

_anomaly_mask = df["Accessory_Max_Size_SqFt"].astype(str).str.contains(
    "Yes|No|Limited", case=False, na=False
)

for _idx in df[_anomaly_mask].index:
    _vals = df.loc[_idx, _shift_window].tolist()
    df.loc[_idx, _shift_window] = [pd.NA] + _vals[:-1]

print(f"Corrected row-misalignment in {_anomaly_mask.sum()} rows "
      f"({df.loc[_anomaly_mask, 'City'].nunique()} town(s): {sorted(df.loc[_anomaly_mask, 'City'].unique())}).")

## 1. Parsing utilities

In [ ]:
def parse_numeric(val):
    """Pull a representative float out of a messy field: plain numbers,
    'A-B' ranges (midpoint), and trailing-plus values like '10+' (base number)."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", "")
    if s == "":
        return np.nan
    m = re.match(r"^(-?\d+\.?\d*)\s*-\s*(-?\d+\.?\d*)$", s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    m = re.match(r"^(-?\d+\.?\d*)\s*\+$", s)
    if m:
        return float(m.group(1))
    m = re.match(r"^(-?\d+\.?\d*)", s)
    if m:
        return float(m.group(1))
    return np.nan


def parse_percent(val):
    """Only returns a number if the field is actually a percentage. Returns NaN
    for lot-shape text like 'Circle 200 ft. diameter' so it's never misread as coverage."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s == "":
        return np.nan
    m = re.search(r"(\d+\.?\d*)\s*%", s)
    if m:
        return float(m.group(1))
    if re.match(r"^\d+\.?\d*$", s):
        return float(s)
    return np.nan


def coalesce(row, cols):
    """First non-null, non-blank value across a set of columns (dimensional fields
    are often duplicated between plain-language and 'E'-coded columns here)."""
    for c in cols:
        v = row.get(c)
        if pd.notna(v) and str(v).strip() != "":
            return v
    return np.nan


def map_use_permission(val):
    """0-100 restrictiveness scale for the composite score (Section 6). 0=freely
    permitted, 100=prohibited. Kept separate from the NZA 3-way classifier below
    because the composite score needs a continuum, not three buckets."""
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    v = str(val).strip().lower()
    if v == "yes":
        return 0
    if "upper" in v:
        return 15
    if v == "no":
        return 100
    if "limited" in v:
        return 55
    if "see note" in v or "see section" in v:
        return 60
    if "special" in v:
        return 65
    return 50


def map_approval_pathway(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    v = str(val).strip().lower()
    if "zoning permit" in v or v == "by right":
        return 0
    if "site plan" in v:
        return 30
    if "special permit" in v:
        return 60
    if "special exception" in v:
        return 70
    if "commission approval" in v:
        return 75
    return 50


def map_yesno_friction(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    v = str(val).strip().lower()
    if v == "yes":
        return 70
    if v == "no":
        return 0
    return np.nan


def infer_approval_pathway(row):
    """When P1 is blank, recover pathway info already embedded inline in a
    use-permission column (e.g. 'Special Exception' written directly in the
    Multifamily column). Returns (mapped_value, was_inferred)."""
    direct = row.get("P1_Approval_Pathway_Multifamily")
    if pd.notna(direct) and str(direct).strip():
        return map_approval_pathway(direct), False
    for c in ["U3_Multifamily_Permitted", "U2_Two_Family_Permitted", "U4_ADU_Permitted"]:
        v = row.get(c)
        if pd.notna(v):
            vv = str(v).strip().lower()
            if "special exception" in vv:
                return 70, True
            if "special permit" in vv:
                return 60, True
            if "commission approval" in vv:
                return 75, True
            if "see note" in vv or "see section" in vv:
                return 60, True
    return np.nan, False


def infer_design_hearing(row, inferred_pathway_val):
    direct = row.get("P2_Design_Hearing_Requirement")
    if pd.notna(direct) and str(direct).strip():
        return map_yesno_friction(direct), False
    if pd.notna(inferred_pathway_val) and inferred_pathway_val >= 70:
        return 70, True
    return np.nan, False

## 2. Scope filter: keep only districts where housing is a legal use

In [ ]:
USE_COLS = [
    "U1_Single_Family_Permitted",
    "U2_Two_Family_Permitted",
    "U3_Multifamily_Permitted",
    "U4_ADU_Permitted",
]


def is_residential_eligible(row):
    def positive(v):
        if pd.isna(v):
            return False
        return str(v).strip().lower() not in ("", "no")

    return any(positive(row[c]) for c in USE_COLS)


mask = df.apply(is_residential_eligible, axis=1)
res = df[mask].copy().reset_index(drop=True)
print(f"{len(res)} of {len(df)} districts pass the residential-use filter "
      f"({df['City'].nunique()} towns in source, {res['City'].nunique()} towns represented after filtering).")

## 3. PRIMARY: the NZA-style scorecard

Three-way classification (Allowed as-of-right / May Be Allowed subject to discretionary review / Prohibited) for each housing type, plus NZA's own minimum-lot-size brackets and a binary parking-mandate flag. This mirrors the structure of NZA's public Jurisdiction Snapshots as closely as our source data allows.

**Known granularity gaps vs. real NZA data, stated plainly rather than papered over:**
- NZA separates **3-Family** from **4+-Family**. Our source data has only a single `Multifamily` column that doesn't make that distinction, so this notebook reports one combined **Multifamily (3+)** category instead of faking a split.
- NZA's ADU classification has 5 buckets (including "with Occupancy Restrictions" variants). Our source data doesn't separately capture occupancy restrictions, so ADU uses the same 3-way bucketing as the other housing types.
- Everything here is **weighted equally per district**, not by land area in acres like NZA — see the note in Section 0.

In [ ]:
def classify_nza_treatment(val):
    """Maps a use-permission cell to NZA's own 3-way vocabulary."""
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown"
    v = str(val).strip().lower()
    if v == "yes" or "upper" in v:
        return "Allowed"
    if v == "no":
        return "Prohibited"
    # Limited / Special Permit / Special Exception / See Note / See Section / anything
    # else conditional -> NZA's "May Be Allowed (Subject to a Public Hearing)" bucket
    return "May Be Allowed"


NZA_HOUSING_TYPES = {
    "1-Family": "U1_Single_Family_Permitted",
    "2-Family": "U2_Two_Family_Permitted",
    "Multifamily (3+)": "U3_Multifamily_Permitted",
    "ADU": "U4_ADU_Permitted",
}

for label, col in NZA_HOUSING_TYPES.items():
    res[f"nza_{label}"] = res[col].apply(classify_nza_treatment)


def bucket_lot_size(row):
    """NZA's own minimum-lot-size brackets for 1-family housing."""
    raw_text = str(row.get("Min_Lot_Size_Area", "")).lower()
    if "varies" in raw_text or "per unit" in raw_text or "per dwelling" in raw_text:
        return "Varies"
    sqft = parse_numeric(row.get("Min_Lot_Size_Area_SqFt"))
    if pd.isna(sqft):
        return "Unknown"
    if sqft == 0:
        return "None"
    if sqft < 10000:
        return "<10,000 sqft"
    if sqft < 20000:
        return "10,000-19,999 sqft"
    if sqft < 40000:
        return "20,000-39,999 sqft"
    if sqft < 80000:
        return "40,000-79,999 sqft"
    return "80,000+ sqft"


res["nza_1family_minlot_bucket"] = res.apply(bucket_lot_size, axis=1)


def has_parking_mandate(val):
    """Binary flag matching NZA's own 'has a minimum parking requirement' stat.
    Any non-blank value (numeric OR text like 'Per Section 6-154') counts as a
    confirmed mandate; blank is 'Unknown' (not researched), never assumed 'No' --
    unlike NZA, we can't be sure a blank cell here means the town confirmed none."""
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown"
    return "Has minimum"


res["nza_parking_mandate"] = res["D2_Min_Off_Street_Parking_Spaces"].apply(has_parking_mandate)

print("NZA-style classification complete.")
res[["City", "Zone"] + [f"nza_{l}" for l in NZA_HOUSING_TYPES] + ["nza_1family_minlot_bucket", "nza_parking_mandate"]].head(5)

## 4. Scorecard report function (district-level or town-level roll-up)

In [ ]:
def nza_scorecard(subset, label):
    """Prints an NZA-style scorecard for a district (subset = 1 row) or a town
    (subset = all that town's filtered districts, equal-weighted per district --
    see Section 0 for why this isn't land-area-weighted)."""
    n = len(subset)
    print(f"{'='*70}")
    print(f"{label}  ({n} district{'s' if n != 1 else ''})")
    print(f"{'='*70}")

    print("\nHousing units allowed (share of districts):")
    for housing_label in NZA_HOUSING_TYPES:
        counts = subset[f"nza_{housing_label}"].value_counts(normalize=True) * 100
        parts = []
        for bucket in ["Allowed", "May Be Allowed", "Prohibited", "Unknown"]:
            pct = counts.get(bucket, 0)
            if pct > 0:
                parts.append(f"{bucket} {pct:.0f}%")
        print(f"  {housing_label:<18} {' | '.join(parts)}")

    print("\n1-family minimum lot size (share of districts):")
    counts = subset["nza_1family_minlot_bucket"].value_counts(normalize=True) * 100
    for bucket in ["None", "<10,000 sqft", "10,000-19,999 sqft", "20,000-39,999 sqft",
                   "40,000-79,999 sqft", "80,000+ sqft", "Varies", "Unknown"]:
        pct = counts.get(bucket, 0)
        if pct > 0:
            print(f"  {bucket:<22} {pct:.0f}%")

    print("\nParking mandate (share of districts):")
    counts = subset["nza_parking_mandate"].value_counts(normalize=True) * 100
    for bucket in ["Has minimum", "Unknown"]:
        pct = counts.get(bucket, 0)
        if pct > 0:
            print(f"  {bucket:<22} {pct:.0f}%")
    print()


greenwich_ra2 = res[(res["City"] == "Greenwich") & (res["Zone"] == "RA-2")]
bridgeport_dx1 = res[(res["City"] == "Bridgeport") & (res["Zone"] == "DX1")]

nza_scorecard(greenwich_ra2, "Greenwich, RA-2")
nza_scorecard(bridgeport_dx1, "Bridgeport, DX1")

## 5. SECONDARY: our composite Restrictiveness Score (0-100)

**This is our own synthesis, not an NZA methodology or an NZA-equivalent score.** It exists because a single ranking number is genuinely useful for prioritizing which districts to target first — but it should never be presented as directly comparable to anything NZA publishes, and the scorecard above is the number that should lead in any external-facing material.

In [ ]:
# --- Index A: Density Restriction ---
res["raw_min_lot_size_sqft"] = res["Min_Lot_Size_Area_SqFt"].apply(parse_numeric)
res["raw_max_density_inv"] = res["D1_Max_Density_Units_Per_Acre"].apply(parse_numeric)
res["raw_max_density_inv"] = -res["raw_max_density_inv"]

# --- Index B: Use Exclusion ---
for c in USE_COLS:
    res[f"raw_{c}"] = res[c].apply(map_use_permission)

# --- Index C: Dimensional Burden ---
res["raw_frontage"] = res["Min_Lot_Size_Frontage"].apply(parse_numeric)
res["raw_front_yard"] = res["Min_Front_Yard_Depth_Feet"].apply(parse_numeric)
res["raw_side_yard"] = res.apply(
    lambda r: parse_numeric(coalesce(r, ["Min_Side_Yard_Width_Feet", "E4_Min_Side_Setback_Ft"])), axis=1
)
res["raw_rear_yard"] = res.apply(
    lambda r: parse_numeric(coalesce(r, ["Min_Rear_Yard_Depth_Feet", "E4_Min_Rear_Setback_Ft"])), axis=1
)
res["raw_far_inv"] = res.apply(
    lambda r: parse_numeric(coalesce(r, ["Floor_Area_Ratio_FAR", "E3_Floor_Area_Ratio_FAR"])), axis=1
)
res["raw_far_inv"] = -res["raw_far_inv"]
res["raw_coverage_inv"] = res["E2_Max_Lot_Coverage_Method"].apply(parse_percent)
res["raw_coverage_inv"] = -res["raw_coverage_inv"]
res["raw_green_area"] = res["Min_Green_Area_Percent"].apply(parse_percent)

# --- Index D: Bulk / Height Cap ---
res["raw_height_stories_inv"] = res.apply(
    lambda r: parse_numeric(coalesce(r, ["Max_Height_Stories", "E1_Max_Height_Stories"])), axis=1
)
res["raw_height_stories_inv"] = -res["raw_height_stories_inv"]
res["raw_height_feet_inv"] = res.apply(
    lambda r: parse_numeric(coalesce(r, ["Max_Height_Feet", "E1_Max_Height_Feet"])), axis=1
)
res["raw_height_feet_inv"] = -res["raw_height_feet_inv"]

# --- Index E: Procedural Friction (with inline-text inference fallbacks) ---
_pathway_results = res.apply(infer_approval_pathway, axis=1)
res["raw_approval_pathway"] = _pathway_results.apply(lambda t: t[0])
res["approval_pathway_inferred"] = _pathway_results.apply(lambda t: t[1])

_hearing_results = [infer_design_hearing(row, row["raw_approval_pathway"]) for _, row in res.iterrows()]
res["raw_design_hearing"] = [t[0] for t in _hearing_results]
res["design_hearing_inferred"] = [t[1] for t in _hearing_results]

res["raw_parking"] = res["D2_Min_Off_Street_Parking_Spaces"].apply(parse_numeric)

CONTINUOUS_RAW = [
    "raw_min_lot_size_sqft", "raw_max_density_inv",
    "raw_frontage", "raw_front_yard", "raw_side_yard", "raw_rear_yard",
    "raw_far_inv", "raw_coverage_inv", "raw_green_area",
    "raw_height_stories_inv", "raw_height_feet_inv",
    "raw_parking",
]
CATEGORICAL_RAW = [
    "raw_U1_Single_Family_Permitted", "raw_U2_Two_Family_Permitted",
    "raw_U3_Multifamily_Permitted", "raw_U4_ADU_Permitted",
    "raw_approval_pathway", "raw_design_hearing",
]
ALL_RAW = [c for c in res.columns if c.startswith("raw_")]
assert set(ALL_RAW) == set(CONTINUOUS_RAW) | set(CATEGORICAL_RAW)

for c in CONTINUOUS_RAW:
    res["pct_" + c[4:]] = res[c].rank(pct=True, na_option="keep") * 100
for c in CATEGORICAL_RAW:
    res["pct_" + c[4:]] = res[c]  # already on a fixed, directionally-correct 0-100 scale

INDEX_DEFINITIONS = {
    "A_Density_Restriction": {"cols": ["pct_min_lot_size_sqft", "pct_max_density_inv"], "weight": 0.30},
    "B_Use_Exclusion": {"cols": [f"pct_{c}" for c in USE_COLS], "weight": 0.25},
    "C_Dimensional_Burden": {
        "cols": ["pct_frontage", "pct_front_yard", "pct_side_yard", "pct_rear_yard",
                 "pct_far_inv", "pct_coverage_inv", "pct_green_area"],
        "weight": 0.20,
    },
    "D_Bulk_Height_Cap": {"cols": ["pct_height_stories_inv", "pct_height_feet_inv"], "weight": 0.10},
    "E_Procedural_Friction": {"cols": ["pct_approval_pathway", "pct_design_hearing", "pct_parking"], "weight": 0.15},
}
assert abs(sum(d["weight"] for d in INDEX_DEFINITIONS.values()) - 1.0) < 1e-9

for idx_name, spec in INDEX_DEFINITIONS.items():
    res[idx_name] = res[spec["cols"]].mean(axis=1, skipna=True)
    res[idx_name + "_n"] = res[spec["cols"]].notna().sum(axis=1)
    res[idx_name + "_confidence"] = res[idx_name + "_n"] / len(spec["cols"])


def weighted_composite(row):
    num, denom = 0.0, 0.0
    for idx_name, spec in INDEX_DEFINITIONS.items():
        val = row[idx_name]
        if pd.notna(val):
            num += val * spec["weight"]
            denom += spec["weight"]
    return num / denom if denom > 0 else np.nan


def composite_confidence(row):
    return sum(spec["weight"] for idx_name, spec in INDEX_DEFINITIONS.items() if pd.notna(row[idx_name]))


res["Restrictiveness_Score"] = res.apply(weighted_composite, axis=1)
res["Restrictiveness_Score_Confidence"] = res.apply(composite_confidence, axis=1)

print(f"Secondary composite score -- Greenwich RA-2: "
      f"{res.loc[(res.City=='Greenwich')&(res.Zone=='RA-2'), 'Restrictiveness_Score'].iloc[0]:.1f}/100")
print(f"Secondary composite score -- Bridgeport DX1: "
      f"{res.loc[(res.City=='Bridgeport')&(res.Zone=='DX1'), 'Restrictiveness_Score'].iloc[0]:.1f}/100")

## 6. Comparison chart: scorecard (primary) + composite (secondary, clearly separated)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [2.2, 1]})

# --- Left panel: PRIMARY -- NZA-style scorecard, grouped by housing type ---
housing_labels = list(NZA_HOUSING_TYPES.keys())
towns = {"Greenwich RA-2": greenwich_ra2, "Bridgeport DX1": bridgeport_dx1}
colors = {"Allowed": "#2980b9", "May Be Allowed": "#aebfd4", "Prohibited": "#c0392b"}
hatches = {"Greenwich RA-2": "", "Bridgeport DX1": "///"}

x = np.arange(len(housing_labels))
width = 0.32
gap = 0.04
for i, (town_label, subset) in enumerate(towns.items()):
    offset = (i - 0.5) * (width + gap)
    bottoms = np.zeros(len(housing_labels))
    for bucket in ["Allowed", "May Be Allowed", "Prohibited"]:
        vals = []
        for h in housing_labels:
            counts = subset[f"nza_{h}"].value_counts(normalize=True) * 100
            vals.append(counts.get(bucket, 0))
        axes[0].bar(x + offset, vals, width, bottom=bottoms, color=colors[bucket],
                    hatch=hatches[town_label], edgecolor="white", linewidth=0.8)
        bottoms += np.array(vals)

axes[0].set_xticks(x)
axes[0].set_xticklabels(housing_labels)
axes[0].set_ylabel("Share of districts (%)")
axes[0].set_ylim(0, 100)
axes[0].set_title("PRIMARY: NZA-style scorecard (by housing type)")

# two-part legend: treatment-type colors, then town hatch pattern
from matplotlib.patches import Patch
treatment_handles = [Patch(facecolor=colors[b], label=b) for b in ["Allowed", "May Be Allowed", "Prohibited"]]
town_handles = [Patch(facecolor="white", edgecolor="black", hatch=hatches[t], label=t) for t in towns]
legend1 = axes[0].legend(handles=treatment_handles, loc="upper center", bbox_to_anchor=(0.5, -0.12),
                          ncol=3, frameon=False, title="Treatment (color)")
axes[0].add_artist(legend1)
axes[0].legend(handles=town_handles, loc="upper center", bbox_to_anchor=(0.5, -0.24),
               ncol=2, frameon=False, title="District (hatch pattern)")

# --- Right panel: SECONDARY -- our composite score, visually de-emphasized ---
comp_vals = [
    res.loc[(res.City == "Greenwich") & (res.Zone == "RA-2"), "Restrictiveness_Score"].iloc[0],
    res.loc[(res.City == "Bridgeport") & (res.Zone == "DX1"), "Restrictiveness_Score"].iloc[0],
]
axes[1].bar(["Greenwich\nRA-2", "Bridgeport\nDX1"], comp_vals, color=["#c0392b", "#2980b9"], alpha=0.55, width=0.5)
axes[1].set_ylim(0, 100)
axes[1].set_title("SECONDARY (our synthesis only):\nComposite Restrictiveness Score", fontsize=10)
axes[1].set_ylabel("0-100 (our own weighting)")
axes[1].axhline(50, color="gray", linestyle="--", linewidth=0.7)

plt.tight_layout()
plt.savefig("../data/collected_data/greenwich_ra2_vs_bridgeport_dx1.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Takeaways

- **Primary (scorecard):** Greenwich RA-2 allows 1-Family outright and prohibits 2-Family, Multifamily, and ADU entirely (100% Prohibited in each, since RA-2 is a single district -- the scorecard function works the same way at a town level, where it becomes a real distribution rather than 0/100 extremes). Bridgeport DX1 allows all four housing types, flagged "Allowed" because its use table lists "Yes (upper)" rather than a discretionary-review phrase -- worth a caveat: upper-floor-only residential in a mixed-use building is a real condition, even though it doesn't trigger our "May Be Allowed" bucket the way a Special Permit would.
- **Secondary (composite):** unchanged from the prior revision -- Greenwich RA-2 scores 68/100, Bridgeport DX1 scores 20/100 on our own synthesized scale. Still useful for ranking many districts at once, but now clearly positioned as our own number, not NZA's.
- **What's still a real limitation:** equal-per-district weighting instead of land-area weighting (Section 0), the 3-Family/4+-Family and ADU-occupancy-restriction granularity gaps (Section 3), and the fact that this only runs against the 25-town structured dataset, not the full 169-town research.
- **Next step for the full deliverable:** the single highest-leverage fix is still merging the other ~144 towns into this same structured schema -- the scorecard and composite logic both already work unchanged against any size of input population.